# LeetCode 1701 — Average Waiting Time bằng Python

Link đề: https://leetcode.com/problems/average-waiting-time/ · Độ khó: **Medium** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def averageWaitingTime(self, customers: List[List[int]]) -> float:
        ...
```

Lộ trình: **mô phỏng từng đơn vị thời gian** (chậm nhưng rất "thật") → **công thức đóng `O(n²)`** cho thời điểm hoàn thành → **mô phỏng theo sự kiện** với một biến `t` (`O(n)`) → **công thức tổng tiền tố + max tiền tố** (`O(n)`, góc nhìn khác). Kết quả là **số thực**, chấp nhận sai số `10⁻⁵`.

## 1. Mục tiêu học

1. Mô hình hóa một **hàng đợi một người phục vụ** (single server queue, FIFO): thời điểm bắt đầu = `max(lúc đến, lúc đầu bếp rảnh)`.
2. Chuyển từ mô phỏng theo **đơn vị thời gian** sang mô phỏng theo **sự kiện** — kỹ thuật cốt lõi trong các bài simulation.
3. Rút ra công thức `finish_i = max(finish_{i−1}, arrival_i) + time_i` và dạng "mở" của nó bằng **tổng tiền tố + max tiền tố**.
4. Làm việc với **số thực**: tránh chia nguyên, so sánh với sai số, dùng `fractions.Fraction` để kiểm tra chính xác.
5. Ước lượng độ lớn số (tổng thời gian chờ ≤ ~`5·10¹³`) để biết có tràn số trong ngôn ngữ khác hay không.

## 2. Đề bài & ví dụ

Một nhà hàng có **một** đầu bếp. Cho mảng `customers`, `customers[i] = [arrival_i, time_i]`:
- `arrival_i` là thời điểm khách thứ `i` đến; mảng đã sắp theo `arrival` **không giảm**.
- `time_i` là thời gian cần để nấu món của khách `i`.

Khi khách đến, khách đưa món cho đầu bếp; đầu bếp bắt đầu nấu **khi rảnh**. Khách chờ tới khi món của mình nấu xong. Đầu bếp nấu **một món tại một thời điểm**, theo **đúng thứ tự** trong input.

Trả về **thời gian chờ trung bình** của mọi khách (thời gian chờ = lúc món xong − lúc đến). Đáp án sai lệch không quá `10⁻⁵` được chấp nhận.

**Ví dụ 1:** `[[1,2],[2,5],[4,3]]` → `5.00000`. Khách 1 chờ `3−1 = 2`, khách 2 chờ `8−2 = 6`, khách 3 chờ `11−4 = 7`; trung bình `(2+6+7)/3 = 5`.

**Ví dụ 2:** `[[5,2],[5,4],[10,3],[20,1]]` → `3.25000`. Thời gian chờ `2, 6, 4, 1`; trung bình `13/4 = 3.25`.

**Ràng buộc:**
- `1 ≤ customers.length ≤ 10⁵`
- `1 ≤ arrival_i, time_i ≤ 10⁴`
- `arrival_i ≤ arrival_{i+1}`

In [1]:
import random
import time
from fractions import Fraction
from typing import List

examples = [
    ([[1, 2], [2, 5], [4, 3]], 5.0),
    ([[5, 2], [5, 4], [10, 3], [20, 1]], 3.25),
]
for cust, want in examples:
    print(f"customers={cust} -> kỳ vọng {want:.5f}")

def close(a, b, eps=1e-5):
    return abs(a - b) <= eps

customers=[[1, 2], [2, 5], [4, 3]] -> kỳ vọng 5.00000
customers=[[5, 2], [5, 4], [10, 3], [20, 1]] -> kỳ vọng 3.25000


## 3. Kiến thức nền

### 3.1 Chia số thực trong Python 3

`a / b` luôn trả về `float`; `a // b` là chia lấy phần nguyên. Bài này cần `/`.

In [2]:
print(13 / 4, 13 // 4, type(13 / 4).__name__)

3.25 3 float


### 3.2 So sánh số thực với sai số

Số thực dấu phẩy động có sai số làm tròn: `0.1 + 0.2 != 0.3`. Khi kiểm tra đáp án, dùng `abs(a − b) ≤ eps` (hoặc `math.isclose`).

In [3]:
import math
print(0.1 + 0.2 == 0.3, abs(0.1 + 0.2 - 0.3) <= 1e-9, math.isclose(0.1 + 0.2, 0.3))

False True True


### 3.3 `fractions.Fraction` — số hữu tỉ chính xác

Dùng để tính "đáp án chuẩn" không sai số khi kiểm thử. `Fraction(13, 4)` giữ nguyên dạng phân số.

In [4]:
x = Fraction(2 + 6 + 7, 3)
print(x, float(x), Fraction(13, 4) == Fraction(26, 8))

5 5.0 True


### 3.4 Giải nén cặp trong vòng `for`

`for a, t in customers:` lấy hai phần tử của mỗi cặp — gọn hơn `c[0]`, `c[1]`.

In [5]:
for a, t in [[1, 2], [2, 5]]:
    print(f"đến lúc {a}, nấu {t}")

đến lúc 1, nấu 2
đến lúc 2, nấu 5


### 3.5 Tổng tiền tố và max tiền tố

`itertools.accumulate(arr)` cho tổng tiền tố; `accumulate(arr, max)` cho max tiền tố. Ta sẽ dùng cả hai ở Cách 4.

In [6]:
from itertools import accumulate
arr = [3, -1, 4, -5, 2]
print("tổng tiền tố:", list(accumulate(arr)))
print("max tiền tố:", list(accumulate(arr, max)))

tổng tiền tố: [3, 2, 6, 1, 3]
max tiền tố: [3, 3, 4, 4, 4]


## 4. Ý tưởng nền tảng / trực giác

Vẽ dòng thời gian cho ví dụ 2 `[[5,2],[5,4],[10,3],[20,1]]`:

| khách | đến | bếp rảnh lúc | bắt đầu = max | xong = bắt đầu + nấu | chờ = xong − đến |
|---|---|---|---|---|---|
| 0 | 5 | 0 | 5 | 7 | 2 |
| 1 | 5 | 7 | 7 | 11 | 6 |
| 2 | 10 | 11 | 11 | 14 | 4 |
| 3 | 20 | 14 | **20** (bếp ngồi chơi 14→20) | 21 | 1 |

Hai tình huống:
- **Bếp đang bận** khi khách đến ⇒ khách phải chờ thêm phần "xếp hàng".
- **Bếp đang rảnh** ⇒ bắt đầu ngay lúc khách đến.

Cả hai gói gọn trong `start = max(arrival, free)`.

## 5. Cách 1 — Brute Force / baseline: mô phỏng từng đơn vị thời gian

**Ý tưởng:** cho đồng hồ chạy `clock = 0, 1, 2, …`. Mỗi tick: nếu bếp rảnh và khách kế tiếp **đã đến**, bắt đầu nấu; nếu đang nấu thì giảm thời gian còn lại; khi nấu xong ghi nhận thời gian chờ. Đúng y như đời thực, nhưng tốn thời gian tỷ lệ với **tổng thời gian** chứ không phải số khách.

In [7]:
def avg_wait_ticks(customers):
    n = len(customers)
    nxt = 0               # khách kế tiếp chưa được nấu
    remaining = 0         # thời gian nấu còn lại của món hiện tại
    serving = -1
    total_wait = 0
    done = 0
    clock = 0
    while done < n:
        if remaining == 0 and nxt < n and customers[nxt][0] <= clock:
            serving = nxt
            remaining = customers[nxt][1]
            nxt += 1
        if remaining > 0:
            remaining -= 1
            if remaining == 0:               # xong ở cuối tick -> thời điểm clock + 1
                total_wait += clock + 1 - customers[serving][0]
                done += 1
                clock += 1
                continue
        clock += 1
    return total_wait / n

for cust, want in examples:
    print(f"{avg_wait_ticks(cust):.5f}", "kỳ vọng", want)

5.00000 kỳ vọng 5.0
3.25000 kỳ vọng 3.25


Lưu ý: sau khi một món xong ở thời điểm `clock + 1`, món tiếp theo có thể bắt đầu ngay tại `clock + 1` — vì vậy ta `continue` để vòng lặp kiểm tra bắt đầu món mới ở tick kế tiếp.

**Độ phức tạp:** Time `O(T)` với `T` = thời điểm món cuối xong ≤ `max(arrival) + Σ time ≤ 10⁴ + 10⁹`. Space `O(1)`.

**Nút thắt:** đồng hồ "đi bộ" qua từng đơn vị, kể cả những khoảng chỉ là chờ đợi. Ta chỉ quan tâm các **sự kiện** (khách đến, món xong).

In [8]:
random.seed(0)
for n in [100, 1000]:
    arr = sorted(random.randint(1, 10**4) for _ in range(n))
    cust = [[a, random.randint(1, 100)] for a in arr]
    t0 = time.perf_counter(); avg_wait_ticks(cust)
    print(f"n={n}, tổng thời gian ≈ {sum(c[1] for c in cust)}: {time.perf_counter() - t0:.3f}s")

n=100, tổng thời gian ≈ 4751: 0.001s
n=1000, tổng thời gian ≈ 50291: 0.003s


## 6. Ý tưởng thuật toán / bước đột phá

**Nhảy thẳng tới sự kiện:** gọi `free` là thời điểm đầu bếp rảnh (ban đầu 0). Xử lý khách `i` theo thứ tự:

- `start_i = max(free, arrival_i)`
- `finish_i = start_i + time_i`
- `wait_i = finish_i − arrival_i`
- `free = finish_i`

**Invariant:** trước khi xử lý khách `i`, `free` = thời điểm món của khách `i−1` xong (hoặc 0), và mọi món trước đều đã hoàn thành theo đúng thứ tự FIFO.

**Dạng "mở" (cho Cách 2 và 4):** khai triển đệ quy `finish_i = max(finish_{i−1}, a_i) + t_i`:

`finish_i = max_{j ≤ i} ( a_j + t_j + t_{j+1} + … + t_i )`

Ý nghĩa: lần gần nhất bếp rảnh trước khi phục vụ khách `i` là ngay trước khách `j` nào đó; từ `j` đến `i` bếp làm liên tục. Lấy max vì khoảng làm liên tục "thật" là khoảng cho thời điểm muộn nhất.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Công thức `max` trên mọi `j ≤ i` (`O(n²)`)

Trực tiếp áp dạng mở: với mỗi `i`, thử mọi `j ≤ i`. Hữu ích để kiểm chứng công thức trước khi tối ưu.

In [9]:
def avg_wait_quadratic(customers):
    n = len(customers)
    total = 0
    for i in range(n):
        best = 0
        s = 0
        for j in range(i, -1, -1):          # cộng dồn t_j + ... + t_i khi j giảm
            s += customers[j][1]
            best = max(best, customers[j][0] + s)
        total += best - customers[i][0]
    return total / n

for cust, want in examples:
    print(f"{avg_wait_quadratic(cust):.5f}", "kỳ vọng", want)

5.00000 kỳ vọng 5.0
3.25000 kỳ vọng 3.25


**Độ phức tạp:** Time `O(n²)`, Space `O(1)`.

### 7.2 Cách 3 — Mô phỏng theo sự kiện với một biến `t` (`O(n)`)

Từng đoạn:
1. `t = 0` (thời điểm bếp rảnh), `total = 0`.
2. Với mỗi `(a, d)`: `t = max(t, a) + d`; `total += t − a`.
3. Trả về `total / n`.

In [10]:
def avg_wait_sim(customers):
    t = 0
    total = 0
    for a, d in customers:
        t = max(t, a) + d
        total += t - a
    return total / len(customers)

for cust, want in examples:
    print(f"{avg_wait_sim(cust):.5f}", "kỳ vọng", want)

5.00000 kỳ vọng 5.0
3.25000 kỳ vọng 3.25


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** Time `O(n)`, Space `O(1)`. `total` là số nguyên (chính xác tuyệt đối), chỉ chia một lần ở cuối ⇒ sai số chỉ đến từ phép chia cuối cùng.

### 7.3 Cách 4 — Tổng tiền tố + max tiền tố (`O(n)`, không có `max(t, a)` tuần tự)

Gọi `T_i = t_0 + … + t_i` (tổng tiền tố thời gian nấu, `T_{−1} = 0`). Khi đó `t_j + … + t_i = T_i − T_{j−1}`, nên

`finish_i = T_i + max_{j ≤ i} ( a_j − T_{j−1} )`.

Phần `max` là **max tiền tố** của dãy `c_j = a_j − T_{j−1}` ⇒ tính một lượt.

In [11]:
def avg_wait_prefix(customers):
    T = 0
    best = float("-inf")
    total = 0
    for a, d in customers:
        best = max(best, a - T)          # c_j = a_j - T_{j-1}
        T += d                            # T bây giờ là T_i
        finish = T + best
        total += finish - a
    return total / len(customers)

for cust, want in examples:
    print(f"{avg_wait_prefix(cust):.5f}", "kỳ vọng", want)

5.00000 kỳ vọng 5.0
3.25000 kỳ vọng 3.25


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Ưu điểm lý thuyết: biểu thức `max` tiền tố có thể song song hóa / dùng cho truy vấn "nếu chỉ phục vụ khách `l..r`" bằng cấu trúc dữ liệu (xem biến thể).

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `[[1,2],[2,5],[4,3]]`:

| khách | a | d | max(t, a) | t mới | chờ = t − a | total |
|---|---|---|---|---|---|---|
| 0 | 1 | 2 | max(0,1) = 1 | 3 | 2 | 2 |
| 1 | 2 | 5 | max(3,2) = 3 | 8 | 6 | 8 |
| 2 | 4 | 3 | max(8,4) = 8 | 11 | 7 | 15 |

`15 / 3 = 5.0`.

In [12]:
def trace_sim(customers):
    t = 0; total = 0
    for i, (a, d) in enumerate(customers):
        start = max(t, a)
        idle = start - t if a > t else 0
        t = start + d; total += t - a
        note = f"(bếp rảnh {idle} đơn vị)" if idle else "(khách phải xếp hàng)" if start > a else ""
        print(f"khách {i}: đến {a:2d}, bắt đầu {start:2d}, xong {t:2d}, chờ {t - a:2d}, total={total:3d} {note}")
    print("trung bình =", total / len(customers))

trace_sim([[1, 2], [2, 5], [4, 3]])

khách 0: đến  1, bắt đầu  1, xong  3, chờ  2, total=  2 (bếp rảnh 1 đơn vị)
khách 1: đến  2, bắt đầu  3, xong  8, chờ  6, total=  8 (khách phải xếp hàng)
khách 2: đến  4, bắt đầu  8, xong 11, chờ  7, total= 15 (khách phải xếp hàng)
trung bình = 5.0


In [13]:
trace_sim([[5, 2], [5, 4], [10, 3], [20, 1]])

khách 0: đến  5, bắt đầu  5, xong  7, chờ  2, total=  2 (bếp rảnh 5 đơn vị)
khách 1: đến  5, bắt đầu  7, xong 11, chờ  6, total=  8 (khách phải xếp hàng)
khách 2: đến 10, bắt đầu 11, xong 14, chờ  4, total= 12 (khách phải xếp hàng)
khách 3: đến 20, bắt đầu 20, xong 21, chờ  1, total= 13 (bếp rảnh 6 đơn vị)
trung bình = 3.25


In [14]:
def trace_prefix(customers):
    T = 0; best = float("-inf")
    for i, (a, d) in enumerate(customers):
        c = a - T
        best = max(best, c); T += d
        print(f"i={i}: c_i = a - T_(i-1) = {a} - {T - d} = {c:3d}, max tiền tố = {best:3d}, T_i = {T:3d}, finish = {T + best}")

trace_prefix([[5, 2], [5, 4], [10, 3], [20, 1]])

i=0: c_i = a - T_(i-1) = 5 - 0 =   5, max tiền tố =   5, T_i =   2, finish = 7
i=1: c_i = a - T_(i-1) = 5 - 2 =   3, max tiền tố =   5, T_i =   6, finish = 11
i=2: c_i = a - T_(i-1) = 10 - 6 =   4, max tiền tố =   5, T_i =   9, finish = 14
i=3: c_i = a - T_(i-1) = 20 - 9 =  11, max tiền tố =  11, T_i =  10, finish = 21


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def averageWaitingTime(self, customers: List[List[int]]) -> float:
        t = 0
        total = 0
        for a, d in customers:
            if t < a:
                t = a
            t += d
            total += t - a
        return total / len(customers)

sol = Solution()
for cust, want in examples:
    got = sol.averageWaitingTime(cust)
    print(f"{got:.5f}", "OK" if close(got, want) else "SAI")

5.00000 OK
3.25000 OK


## 10. Vì sao thuật toán đúng

**Cách 3.** Quy nạp theo `i`. Giả sử sau khi xử lý khách `i−1`, `t = finish_{i−1}` đúng. Đầu bếp chỉ bắt đầu món `i` khi (1) đã xong món `i−1` (FIFO, một món mỗi lúc) và (2) khách `i` đã đến. Không có lý do gì để chờ thêm ⇒ `start_i = max(finish_{i−1}, a_i)`, `finish_i = start_i + t_i`. ✓ Thời gian chờ là `finish_i − a_i` theo định nghĩa.

**Dạng mở (Cách 2, 4).** Quy nạp: `finish_0 = a_0 + t_0`. Giả sử `finish_{i−1} = max_{j ≤ i−1} (a_j + T_{i−1} − T_{j−1})`. Khi đó

`finish_i = max(finish_{i−1}, a_i) + t_i = max( max_{j≤i−1}(a_j + T_{i−1} − T_{j−1}) + t_i,  a_i + t_i ) = max_{j ≤ i} (a_j + T_i − T_{j−1})`. ✓

**Độ chính xác số thực.** `total` là số nguyên chính xác; phép chia `total / n` cho kết quả `float` gần nhất với giá trị thật (sai số tương đối ~`10⁻¹⁶`) — rất xa ngưỡng `10⁻⁵`.

Lưu ý: điều kiện `arrival` không giảm đảm bảo thứ tự input cũng là thứ tự đến; công thức vẫn đúng dù arrival không sắp xếp miễn là đầu bếp phục vụ theo thứ tự input.

## 11. Phân tích Time / Space

Gọi `n = len(customers) ≤ 10⁵`, `T_end` = thời điểm món cuối xong ≤ `10⁴ + 10⁵ · 10⁴ ≈ 10⁹`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng tick | `O(T_end)` | `O(1)` | ~10⁹ bước worst — quá chậm |
| 2. Công thức `O(n²)` | `O(n²)` | `O(1)` | kiểm chứng công thức |
| 3. Sự kiện | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tiền tố + max tiền tố | `O(n)` | `O(1)` | góc nhìn đại số |

**Độ lớn số:** mỗi thời gian chờ ≤ `T_end ≈ 10⁹`, tổng ≤ `10⁵ · 10⁹ = 10¹⁴` — vượt `int32`, cần `long` trong C++/Java; Python không lo tràn.

In [16]:
worst = [[1, 10**4] for _ in range(10**5)]      # mọi khách đến lúc 1, mỗi món 10^4
t0 = time.perf_counter(); r = Solution().averageWaitingTime(worst)
total = sum((i + 1) * 10**4 for i in range(10**5))
print(f"trung bình = {r:.1f} | tổng chờ = {total} (> 2^31 = {2**31}) | {time.perf_counter() - t0:.3f}s")

trung bình = 500005000.0 | tổng chờ = 50000500000000 (> 2^31 = 2147483648) | 0.016s


## 12. Bộ test

Đáp án chuẩn tính bằng `Fraction` (chính xác), so với mọi cách bằng sai số `10⁻⁵`. Dữ liệu ngẫu nhiên đúng ràng buộc: `arrival` không giảm, `1 ≤ arrival, time ≤ 10⁴` (với mô phỏng tick, dùng giá trị nhỏ cho nhanh).

In [17]:
def avg_wait_exact(customers):
    t = 0; total = 0
    for a, d in customers:
        t = max(t, a) + d; total += t - a
    return Fraction(total, len(customers))

def gen(n, amax, tmax, rng):
    arr = sorted(rng.randint(1, amax) for _ in range(n))
    return [[a, rng.randint(1, tmax)] for a in arr]

fns = [avg_wait_ticks, avg_wait_quadratic, avg_wait_sim, avg_wait_prefix,
       lambda c: Solution().averageWaitingTime(c)]

for cust, want in examples:
    for fn in fns:
        assert close(fn(cust), want)

edge = [
    ([[1, 1]], 1.0),                              # một khách
    ([[10000, 10000]], 10000.0),
    ([[1, 5], [1, 5], [1, 5]], (5 + 10 + 15) / 3),  # cùng lúc đến
    ([[1, 1], [100, 1], [1000, 1]], 1.0),         # bếp luôn rảnh
]
for cust, want in edge:
    for fn in fns:
        assert close(fn(cust), want), (fn, cust)

rng = random.Random(2024)
for _ in range(500):
    cust = gen(rng.randint(1, 30), rng.choice([5, 50, 300]), rng.choice([3, 20]), rng)
    exact = avg_wait_exact(cust)
    for fn in fns:
        assert abs(Fraction(fn(cust)) - exact) <= Fraction(1, 10**5), (fn, cust)
print("ví dụ + edge + 500 bộ ngẫu nhiên (so với Fraction): OK")

ví dụ + edge + 500 bộ ngẫu nhiên (so với Fraction): OK


In [18]:
rng = random.Random(99)
for _ in range(20):
    cust = gen(rng.randint(1000, 5000), 10**4, 10**4, rng)
    exact = avg_wait_exact(cust)
    for fn in [avg_wait_sim, avg_wait_prefix, Solution().averageWaitingTime]:
        assert abs(Fraction(fn(cust)) - exact) <= Fraction(1, 10**5)
print("n tới 5000, giá trị tới 10^4: các cách O(n) chính xác trong 1e-5")

n tới 5000, giá trị tới 10^4: các cách O(n) chính xác trong 1e-5


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Luôn bắt đầu nấu lúc khách đến (quên bếp đang bận)

In [19]:
def wrong_ignore_busy(customers):
    total = sum(d for a, d in customers)      # SAI: coi như chỉ chờ thời gian nấu
    return total / len(customers)

for cust, want in examples:
    print(f"sai = {wrong_ignore_busy(cust):.5f} | đúng = {want:.5f}")

sai = 3.33333 | đúng = 5.00000
sai = 2.50000 | đúng = 3.25000


**Sửa:** `start = max(t, a)`.

### Lỗi 2 — Luôn bắt đầu khi bếp vừa xong món trước (quên bếp có thể rảnh, khách chưa đến)

Viết `t += d` mà không `max(t, a)`: bếp "nấu trước" cho khách chưa đến — thời gian chờ có thể **âm**.

In [20]:
def wrong_ignore_idle(customers):
    t = 0; total = 0
    for a, d in customers:
        t += d                                 # SAI: thiếu t = max(t, a)
        total += t - a
    return total / len(customers)

for cust, want in examples:
    print(f"sai = {wrong_ignore_idle(cust):.5f} | đúng = {want:.5f}")

sai = 4.00000 | đúng = 5.00000
sai = -3.25000 | đúng = 3.25000


**Sửa:** cập nhật `t = max(t, a)` trước khi cộng `d`.

### Lỗi 3 — Tính thời gian chờ là thời gian **xếp hàng** (bỏ phần nấu)

Đề định nghĩa chờ = tới khi **món xong**, tức gồm cả thời gian nấu.

In [21]:
def wrong_queue_only(customers):
    t = 0; total = 0
    for a, d in customers:
        start = max(t, a)
        total += start - a                     # SAI: thiếu + d
        t = start + d
    return total / len(customers)

for cust, want in examples:
    print(f"sai = {wrong_queue_only(cust):.5f} | đúng = {want:.5f}")

sai = 1.66667 | đúng = 5.00000
sai = 0.75000 | đúng = 3.25000


**Sửa:** `total += finish − a` với `finish = start + d`.

### Lỗi 4 — Chia nguyên `//` (hoặc tính trung bình dồn từng bước bằng số nguyên)

In [22]:
def wrong_int_div(customers):
    t = 0; total = 0
    for a, d in customers:
        t = max(t, a) + d; total += t - a
    return total // len(customers)            # SAI: mất phần thập phân

print("ví dụ 2: sai =", wrong_int_div(examples[1][0]), "| đúng =", examples[1][1])

ví dụ 2: sai = 3 | đúng = 3.25


**Sửa:** dùng `/`. Trong C++/Java nhớ ép kiểu `(double) total / n` và dùng `long` cho `total`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Hàng đợi một người phục vụ, FIFO** ⇒ một biến "thời điểm rảnh" + `max(rảnh, đến)`.
- Mô phỏng theo **sự kiện** thay vì theo đơn vị thời gian khi thời gian lớn.
- Giữ tổng là **số nguyên**, chỉ chia ở cuối ⇒ tránh tích lũy sai số.
- Ước lượng biên độ số để chọn kiểu (`long`) ở ngôn ngữ có tràn số.
- Cùng họ: *Single-Threaded CPU* (1834, phải chọn việc ngắn nhất — cần heap), *Process Tasks Using Servers* (1882, nhiều server — heap), *Task Scheduler*…

## 15. Biến thể của bài

1. **Nhiều đầu bếp (k server), vẫn FIFO:** khách được giao cho đầu bếp rảnh sớm nhất ⇒ min-heap thời điểm rảnh.
2. **Đầu bếp chọn món ngắn nhất trong các khách đã đến** (Shortest Job First) ⇒ LeetCode 1834 Single-Threaded CPU — heap theo `(time, index)`.
3. **Trả về thời gian chờ lớn nhất / trung vị:** lưu danh sách `wait_i`.
4. **Truy vấn: nếu nhà hàng chỉ phục vụ khách `l..r`** thì tổng chờ? Dùng dạng mở `finish_i = T_i + max_{l ≤ j ≤ i}(a_j − T_{j−1})` (max trên đoạn, cần sparse table / stack đơn điệu offline).
5. **Bếp nghỉ giữa các món `b` đơn vị:** `t = max(t + b, a) + d` (cẩn thận món đầu).

In [23]:
import heapq

def avg_wait_k_chefs(customers, k):
    """Biến thể 1: k đầu bếp, khách giao cho đầu bếp rảnh sớm nhất (FIFO)."""
    free = [0] * k
    total = 0
    for a, d in customers:
        f = heapq.heappop(free)
        finish = max(f, a) + d
        total += finish - a
        heapq.heappush(free, finish)
    return total / len(customers)

cust = [[5, 2], [5, 4], [10, 3], [20, 1]]
for k in [1, 2, 4]:
    print(f"k={k} đầu bếp: trung bình chờ = {avg_wait_k_chefs(cust, k):.5f}")
assert close(avg_wait_k_chefs(cust, 1), 3.25)

k=1 đầu bếp: trung bình chờ = 3.25000
k=2 đầu bếp: trung bình chờ = 2.50000
k=4 đầu bếp: trung bình chờ = 2.50000


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng tick | chạy đồng hồ từng đơn vị | `O(T_end)` | `O(1)` | hiểu bài, kiểm chứng với số nhỏ |
| 2. Công thức `O(n²)` | `max_j (a_j + Σ t)` | `O(n²)` | `O(1)` | kiểm chứng dạng mở |
| 3. Sự kiện | `t = max(t, a) + d` | `O(n)` | `O(1)` | **nộp bài** |
| 4. Tiền tố + max | `T_i + max(a_j − T_{j−1})` | `O(n)` | `O(1)` | biến thể truy vấn đoạn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `averageWaitingTime([[2, 3], [6, 3], [7, 5], [11, 3], [15, 2], [18, 1]])`?

**Bài 2 (tính tay):** với `[[1, 4], [2, 1], [3, 1], [10, 2]]`, lập bảng `start`, `finish`, `wait`.

**Bài 3 (điền TODO):** hoàn thiện `idle_time(customers)` trả về **tổng thời gian đầu bếp ngồi chơi** từ thời điểm 0 tới khi món cuối xong.

**Bài 4 (tự viết lại):** viết `max_wait(customers)` trả về thời gian chờ lớn nhất, dùng Cách 4 (tiền tố + max tiền tố).

**Bài 5 (mini challenge):** biến thể 2 — đầu bếp **chọn món ngắn nhất** trong các khách đã đến (hòa thì chọn khách đến trước / chỉ số nhỏ); tính thời gian chờ trung bình. Kiểm chứng bằng mô phỏng tick. (Biến thể tự đặt, gần LeetCode 1834.)

In [24]:
def idle_time(customers):
    # TODO: cộng dồn max(0, a - t) trước khi cập nhật t
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Mô phỏng `t`. (2) Thời gian chờ lần lượt `3, 3, 7, 6, 4, 2`. (3) Tổng 25, trung bình `25/6 ≈ 4.16667`.

**Bài 2:** (1) Khách 0: `1→5`. (2) Khách 1: `5→6`, khách 2: `6→7`. (3) Khách 3: bếp rảnh `7→10`, `10→12`; chờ `4, 4, 4, 2`.

**Bài 3:** (1) Bếp ngồi chơi khi `a > t`. (2) Cộng `a − t` khi đó. (3) Tính cả khoảng `0 → a_0`.

**Bài 4:** (1) `wait_i = T_i + best − a_i`. (2) Lấy max trong vòng lặp. (3) Đối chiếu Cách 3.

**Bài 5:** (1) Sắp các khách theo `arrival` (đã sắp). (2) Khi bếp rảnh, đẩy mọi khách đã đến vào heap `(time, index)`; nếu heap rỗng thì nhảy `t` tới `arrival` kế tiếp. (3) Lấy món nhỏ nhất, cập nhật `t`, cộng thời gian chờ.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
r = Solution().averageWaitingTime([[2, 3], [6, 3], [7, 5], [11, 3], [15, 2], [18, 1]])
assert close(r, 25 / 6)

# Bài 2
def table(customers):
    t = 0; rows = []
    for a, d in customers:
        s = max(t, a); t = s + d; rows.append((s, t, t - a))
    return rows
assert table([[1, 4], [2, 1], [3, 1], [10, 2]]) == [(1, 5, 4), (5, 6, 4), (6, 7, 4), (10, 12, 2)]

# Bài 3
def idle_time_ans(customers):
    t = 0; idle = 0
    for a, d in customers:
        if a > t:
            idle += a - t; t = a
        t += d
    return idle
assert idle_time_ans([[5, 2], [5, 4], [10, 3], [20, 1]]) == 5 + 6     # 0->5 và 14->20

# Bài 4
def max_wait(customers):
    T = 0; best = float("-inf"); ans = 0
    for a, d in customers:
        best = max(best, a - T); T += d
        ans = max(ans, T + best - a)
    return ans

rng = random.Random(41)
for _ in range(300):
    cust = gen(rng.randint(1, 40), 100, 20, rng)
    assert max_wait(cust) == max(w for _, _, w in table(cust))
    # tổng: thời điểm xong cuối = tổng nấu + tổng ngồi chơi
    assert table(cust)[-1][1] == sum(d for _, d in cust) + idle_time_ans(cust)
print("Bài 1-4: OK")

Bài 1-4: OK


In [26]:
# Bài 5 — Shortest Job First
def avg_wait_sjf(customers):
    n = len(customers); h = []; i = 0; t = 0; total = 0; done = 0
    while done < n:
        if not h and t < customers[i][0]:
            t = customers[i][0]
        while i < n and customers[i][0] <= t:
            heapq.heappush(h, (customers[i][1], i)); i += 1
        d, j = heapq.heappop(h)
        t += d; total += t - customers[j][0]; done += 1
    return total / n

def avg_wait_sjf_ticks(customers):
    n = len(customers); waiting = []; i = 0; clock = 0; cur = None; rem = 0; total = 0; done = 0
    while done < n:
        while i < n and customers[i][0] <= clock:
            waiting.append(i); i += 1
        if rem == 0 and waiting:
            cur = min(waiting, key=lambda j: (customers[j][1], j)); waiting.remove(cur); rem = customers[cur][1]
        clock += 1
        if rem > 0:
            rem -= 1
            if rem == 0:
                total += clock - customers[cur][0]; done += 1
    return total / n

rng = random.Random(42)
for _ in range(300):
    cust = gen(rng.randint(1, 15), 30, 6, rng)
    assert close(avg_wait_sjf(cust), avg_wait_sjf_ticks(cust)), cust
print("Bài 5: SJF khớp mô phỏng tick; ví dụ 1 FIFO =", Solution().averageWaitingTime(examples[0][0]),
      "| SJF =", round(avg_wait_sjf(examples[0][0]), 5))

Bài 5: SJF khớp mô phỏng tick; ví dụ 1 FIFO = 5.0 | SJF = 5.0


## 20. Template nhớ nhanh + Checklist

```python
# Hàng đợi 1 người phục vụ, FIFO
t = 0; total = 0
for a, d in customers:
    t = max(t, a) + d        # bắt đầu khi cả bếp rảnh lẫn khách đã đến
    total += t - a           # chờ = xong - đến (gồm cả thời gian nấu)
return total / len(customers)
```

**Checklist trước khi submit:**
- [ ] `max(t, a)` — xử lý cả hai trường hợp bếp bận / bếp rảnh.
- [ ] Thời gian chờ tính tới lúc **món xong** (gồm thời gian nấu).
- [ ] Dùng `/` (số thực), chỉ chia một lần ở cuối.
- [ ] (C++/Java) `total` kiểu `long` — tổng có thể tới ~`5·10¹³`.
- [ ] Thử một khách, nhiều khách đến cùng lúc, bếp luôn rảnh.